# LLMBackend Pickup Roles

This notebook mirrors the KRROOD probabilistic pickup-parameterization test, but uses `llmr.backend.LLMBackend` instead.

It demonstrates two roles:

1. Natural-language instruction -> classify action -> LLMBackend resolves a `Match(PickUpAction)`.
2. Existing pytest-style underspecified `PickUpAction` -> `context.query_backend = LLMBackend(...)` resolves the free slots.

Both roles preserve the LLM-selected pickup parameters. PyCRAM only computes a reachable navigation/base pose before executing pickup.


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body

# print("World loaded:", world)
# print("Robot:", robot_view)
# print("Instruction:", INSTRUCTION)


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [2]:
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))


LLM ready: gpt-4o-mini


In [3]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [10]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


## Shared PyCRAM Helpers

`LLMBackend` resolves the pickup action parameters. PyCRAM still needs a reachable robot base pose before pickup can execute, so the helper below computes a `CostmapLocation` using the exact arm and grasp chosen by the LLM.

All motion execution must run inside `simulated_robot`; otherwise PyCRAM has no execution backend for motions and posture changes will not affect reachability.


In [5]:
from pycram.datastructures.enums import Arms
from pycram.exceptions import MotionDidNotFinish
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState


def prepare_robot_for_pickup(context):
    setup_plan = sequential(
        [
            ParkArmsAction(Arms.BOTH),
            MoveTorsoAction(TorsoState.HIGH),
        ],
        context=context,
    )
    with simulated_robot:
        setup_plan.perform()


def build_navigate_then_pick(action_instance, context):
    pickup_pose = CostmapLocation(
        target=action_instance.object_designator.global_pose,
        reachable=True,
        reachable_arm=action_instance.arm,
        grasp_description=action_instance.grasp_description,
        context=context,
    ).ground()

    return sequential(
        [
            NavigateAction(pickup_pose, keep_joint_states=True),
            PickUpAction(
                object_designator=action_instance.object_designator,
                arm=action_instance.arm,
                grasp_description=action_instance.grasp_description,
            ),
        ],
        context=context,
    )


def print_pickup_action(action_instance):
    print("Action type:", type(action_instance).__name__)
    print("Object     :", action_instance.object_designator)
    print("Arm        :", action_instance.arm)
    print("Grasp      :", action_instance.grasp_description)
    print("Offset     :", action_instance.grasp_description.manipulation_offset)


## Role 1 — Natural-Language Instruction

This role starts with a natural-language command. The LLM first classifies the action class, then `instance_from_match()` invokes `LLMBackend` to resolve a query-style `Match(PickUpAction)`.

Complex required fields are represented as nested `Match(...)` objects, so KRROOD owns recursive construction while `LLMBackend` only fills the free leaf slots.


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr.reasoning.slot_filler import infer_action_class
from llmr import instance_from_match
from pycram.datastructures.grasp import GraspDescription

INSTRUCTION = "pick up the milk from the table"

role1_classification = infer_action_class(
    instruction=INSTRUCTION,
    llm=llm,
)
assert role1_classification is not None, "LLM could not classify the instruction"

# infer_action_class returns the raw ActionClassificationResult; look up the class using the
# same registry the classifier sees (auto-discovered here by the pycram bridge).
from llmr.pycram import discover_action_classes
role1_action_cls = discover_action_classes()[role1_classification.action_type]

print("Classified action:", role1_action_cls)
assert role1_action_cls is PickUpAction

role1_pick_query = Match(role1_action_cls)(
    object_designator=...,
    arm=...,
    grasp_description=Match(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        manipulator=...,
    ),
)

role1_action_instance = instance_from_match(
    match=role1_pick_query,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)


In [ ]:
# role1_action_instance

In [ ]:
prepare_robot_for_pickup(context)

role1_plan_node = build_navigate_then_pick(role1_action_instance, context)
print("Role 1 Navigate + PickUp plan ready:", role1_plan_node)


In [ ]:
with simulated_robot:
    try:
        role1_plan_node.perform()
    except MotionDidNotFinish:
        print("Role 1 motion did not finish; pickup parameters and plan construction were still exercised.")


## Role 2 — Existing Underspecified Instance

This role mirrors the pytest-style KRROOD query. The object and `manipulation_offset` are fixed, while `arm`, grasp enums, and `rotate_gripper` remain LLM-filled slots. `manipulator` is represented as a KRROOD variable over semantic annotations, matching the original parameterization-style query.

Instead of `ProbabilisticBackend`, the context uses `LLMBackend`.


In [6]:
from krrood.entity_query_language.factories import underspecified, variable, variable_from
from krrood.parametrization.parameterizer import UnderspecifiedParameters
from llmr.backend import LLMBackend
from pycram.datastructures.grasp import GraspDescription
from semantic_digital_twin.robots.abstract_robot import Manipulator

INSTRUCTION = "pick up the milk from the table"

milk = world.get_body_by_name("milk.stl")
milk_variable = variable_from([milk])

role2_pick_up_description = underspecified(PickUpAction)(
    object_designator=milk_variable,
    arm=...,
    grasp_description=underspecified(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        rotate_gripper=...,
        manipulation_offset=0.05,
        manipulator=variable(Manipulator, world.semantic_annotations),
    ),
)
role2_pick_up_description.resolve()

parameters = UnderspecifiedParameters(role2_pick_up_description)
print("Variables:", [v.name for v in parameters.variables.values()])
print("Variable count:", len(parameters.variables))

[manipulation_offset_var] = [
    v for v in parameters.variables.values() if v.name.endswith("manipulation_offset")
]
print("Fixed manipulation_offset:", parameters.assignments_for_conditioning[manipulation_offset_var])
assert parameters.assignments_for_conditioning[manipulation_offset_var] == 0.05


Variables: ['PickUpAction.object_designator', 'PickUpAction.arm', 'PickUpAction.grasp_description.approach_direction', 'PickUpAction.grasp_description.vertical_alignment', 'PickUpAction.grasp_description.rotate_gripper', 'PickUpAction.grasp_description.manipulation_offset', 'PickUpAction.grasp_description.manipulator']
Variable count: 7
Fixed manipulation_offset: 0.05


In [7]:
# parameters.variables

{'PickUpAction.object_designator': Symbolic(PickUpAction.object_designator),
 'PickUpAction.arm': Symbolic(PickUpAction.arm),
 'PickUpAction.grasp_description.approach_direction': Symbolic(PickUpAction.grasp_description.approach_direction),
 'PickUpAction.grasp_description.vertical_alignment': Symbolic(PickUpAction.grasp_description.vertical_alignment),
 'PickUpAction.grasp_description.rotate_gripper': Symbolic(PickUpAction.grasp_description.rotate_gripper),
 'PickUpAction.grasp_description.manipulation_offset': Continuous(PickUpAction.grasp_description.manipulation_offset),
 'PickUpAction.grasp_description.manipulator': Symbolic(PickUpAction.grasp_description.manipulator)}

In [8]:
context.query_backend = LLMBackend(
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

role2_action_instance = next(iter(context.query_backend.evaluate(role2_pick_up_description)))
print_pickup_action(role2_action_instance)


Action type: PickUpAction
Object     : Body(name=PrefixedName('None/milk.stl'), id=UUID('836fda69-94de-4358-b3b7-37a01c607a8e'), index=219)
Arm        : RIGHT
Grasp      : GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/left_gripper'), id=UUID('eec9d2dd-9acd-4946-b86e-c68f83f89c45'), root=Body(name=PrefixedName('pr2/l_gripper_palm_link'), id=UUID('33e814a6-2466-4be9-b618-5d5ae87d5a5b'), index=84), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('a4b3ab9b-adac-481e-a85a-f5e214dfe74b'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('0f6263b5-df12-473e-a5f6-7b96ba6733a0'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('ea994c17-2081-4e8b-b496-4fdd91421998'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_ste

In [9]:
role2_action_instance.arm

RIGHT

In [ ]:
prepare_robot_for_pickup(context)

role2_plan_node = build_navigate_then_pick(role2_action_instance, context)
print("Role 2 Navigate + PickUp plan ready:", role2_plan_node)


In [ ]:
with simulated_robot:
    try:
        role2_plan_node.perform()
    except MotionDidNotFinish:
        print("Role 2 motion did not finish; LLMBackend query resolution and plan construction were still exercised.")
